In [1]:
# Check and fill any remaining NaN values
print("NaN counts before fix:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

# Fill remaining NaNs with median
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())

print("\nNaN counts after fix:")
print(X_train.isnull().sum().sum(), "total NaNs remaining")

NaN counts before fix:


NameError: name 'X_train' is not defined

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    confusion_matrix, roc_curve
)
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/kaggle/input/datasets/maryamrafaqat/data-engineered/data_engineered.csv')

TARGET = 'SeriousDlqin2yrs'
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
scale = (y_train == 0).sum() / (y_train == 1).sum()

In [ ]:
def get_metrics(y_true, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    ks, _ = ks_2samp(probs[y_true == 0], probs[y_true == 1])
    return {
        'AUC-ROC': round(roc_auc_score(y_true, probs), 4),
        'PR-AUC':  round(average_precision_score(y_true, probs), 4),
        'KS':      round(ks, 4),
        'F1':      round(f1_score(y_true, preds), 4),
        'FNR':     round(fn / (fn + tp), 4),
    }


models = {
    'Logistic Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
    ]),
    'XGBoost': XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        scale_pos_weight=scale, subsample=0.8,
        colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbosity=0
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        scale_pos_weight=scale, subsample=0.8,
        colsample_bytree=0.8, random_state=42,
        n_jobs=-1, verbosity=-1
    ),
}
results = {}
roc_data = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    cal = CalibratedClassifierCV(model, method='isotonic', cv='prefit')
    cal.fit(X_test, y_test)

    probs = cal.predict_proba(X_test)[:, 1]
    results[name] = get_metrics(y_test, probs)

    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_data[name] = (fpr, tpr, results[name]['AUC-ROC'])
    print(f"  Done. AUC-ROC: {results[name]['AUC-ROC']}")

In [ ]:
# Results table
results_df = pd.DataFrame(results).T
print("\n=== Model Comparison ===")
print(results_df.to_string())

# Highlight best in each column
results_df.style.highlight_max(axis=0, color='lightgreen') \
               .highlight_min(subset=['FNR'], axis=0, color='lightgreen')

In [ ]:
# ROC curves
plt.figure(figsize=(8, 6))
colors = ['steelblue', 'tomato', 'green']
for (name, (fpr, tpr, auc)), color in zip(roc_data.items(), colors):
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc})', color=color)
plt.plot([0,1],[0,1],'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics_to_plot = ['AUC-ROC', 'PR-AUC', 'KS']
colors = ['steelblue', 'tomato', 'green']

for i, metric in enumerate(metrics_to_plot):
    values = results_df[metric]
    bars = axes[i].bar(values.index, values.values, color=colors, edgecolor='black')
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].set_xticklabels(values.index, rotation=15)
    for bar, val in zip(bars, values.values):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.01,
                     f'{val:.4f}', ha='center', fontsize=9)

plt.suptitle('Model Performance Comparison', fontsize=13)
plt.tight_layout()
plt.show()